## Import Account Securely

In [ ]:
# dotenv to securely interact with API keys in .env file
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("API_KEY")
api_secret = os.getenv("SECRET_KEY")

In [ ]:
from binance.client import Client
import pandas as pd

In [ ]:
client = Client(api_key = api_key, api_secret = api_secret, tld = "com")

In [ ]:
# Example of account information retrieval, more on docs https://python-binance.readthedocs.io/en/latest/account.html
balances = client.get_account()["balances"] 
balances = pd.DataFrame(balances)

# Print only assets with non-dust values
cols = ["free", "locked"]
balances[cols] = balances[cols].apply(pd.to_numeric, errors="coerce")
print(balances[balances["free"] > 1])

## Historical Data

In [ ]:
# Fetching earliest timestamp
timestamp = client._get_earliest_valid_timestamp(symbol = "BTCUSDT", interval="1d")

In [ ]:
bars = client.get_historical_klines(symbol = "BTCUSDT", interval = "1d", start_str = timestamp, limit = 1000)
df = pd.DataFrame(bars)

In [ ]:
df.columns = ["Open time", "Open", "High", "Low", "Close", "Volume", "Close time", "Quote asset volume", "Number of trades", "Taker buy base asset volume", "Taker buy quote asset volume", "Ignore"]

In [ ]:
df["Date"] = pd.to_datetime(df["Open time"], unit='ms')

In [ ]:
df.set_index("Date", inplace = True)

In [ ]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors = "coerce")

print(df.dtypes)

### Refactoring into a function

In [ ]:
def get_binance_ohlcv(symbol, interval, start, end=None):
    bars = client.get_historical_klines(symbol, interval, start, end)
    df = pd.DataFrame(bars)
                      
    df.columns = ["Open time", "Open", "High", "Low", "Close", "Volume", "Close time", "Quote asset volume", "Number of trades", "Taker buy base asset volume", "Taker buy quote asset volume", "Ignore"]

    df["Date"] = pd.to_datetime(df["Open time"], unit='ms')
    df.set_index("Date", inplace = True)

    df = df[["Open", "High", "Low", "Close", "Volume"]].copy()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors = "coerce")

    return df

In [ ]:
df = get_binance_ohlcv("BTCUSDT", "1d", "2021-05-05")
df

## Streaming Data

In [ ]:
import asyncio
from binance import AsyncClient, BinanceSocketManager

In [ ]:
async def trade_stream():
    client = await AsyncClient.create()
    bm = BinanceSocketManager(client)
    ts = bm.trade_socket('BTCUSDT')
    async with ts as tscm:
        for _ in range(50): 
            res = await tscm.recv()
            print(res)

    await client.close_connection()

In [ ]:
await trade_stream()

In [ ]:
async def kline_stream():
    client = await AsyncClient.create()
    bm = BinanceSocketManager(client)
    ks = bm.kline_socket('BTCUSDT')
    async with ks as kscm:
        for _ in range(50): 
            res = await kscm.recv()
            print(res)

    await client.close_connection()

In [ ]:
await kline_stream()

In [ ]:
# Interesting experiment of async, but not entirely useful to have the sockets collated in one function

async def stream_data():
    client = await AsyncClient.create()
    bm = BinanceSocketManager(client)

    async def trade_socket(bm): 
        ts = bm.trade_socket('BTCUSDT')
        async with ts as tscm:
            for _ in range(50): 
                res = await tscm.recv()
                print(res)

    async def kline_socket(bm): 
        ks = bm.kline_socket('BTCUSDT')
        async with ks as kscm:
            for _ in range(50): 
                res = await kscm.recv()
                print(res)

    await asyncio.gather(
        trade_socket(bm),
        kline_socket(bm)
    )
    
    await client.close_connection()

In [ ]:
# Running all stream asynchronously

async def stream_all():

    await asyncio.gather(
        trade_stream(),
        kline_stream()
    )

    await client.close_connection()

In [ ]:
await stream_all()

### Experimenting with callback function

In [ ]:
def print_kline_cb(res):
    print(res['k']['o'])

In [ ]:
async def kline_stream(cb):
    client = await AsyncClient.create()
    bm = BinanceSocketManager(client)
    ks = bm.kline_socket('BTCUSDT')
    async with ks as kscm:
        for _ in range(50): 
            res = await kscm.recv()
            cb(res)

    await client.close_connection()

In [ ]:
await kline_stream(print_kline_cb)

### Experimenting with variable scope and callback function

In [ ]:
async def core_kline(bm, cb):
    ks = bm.kline_socket('BTCUSDT')
    async with ks as kscm:
        for _ in range(10):
            res = await kscm.recv()
            cb(res)

In [ ]:
async def run_core():
    client = await AsyncClient.create()
    bm = BinanceSocketManager(client)
    await core_kline(bm, print_kline_cb)
    await client.close_connection()

In [ ]:
await run_core()